# Sahayak - Personal object recogniser on ORBIT (transfer learning, Kaggle)
This is the **novel part** of the project: a few-shot recogniser for a blind user's *own* objects. A DINOv2 backbone is
fine-tuned with metric learning (last 2 blocks / LoRA) on ORBIT, a dataset filmed by blind and low-vision people, then evaluated
on **users it has never seen** with unknown objects mixed in.

**Before you start:** right panel -> Session options -> Accelerator -> **GPU T4 x2** (not P100), and Internet **On**. Run cells top to bottom.
**Honest status:** this code passes its unit tests and a synthetic smoke run, but has NOT been run on real ORBIT data yet.
The converter was written from ORBIT's documented layout. Check the dry-run output (step 5) before training.

In [ ]:
!nvidia-smi | head -12

## 1. Project + packages

In [ ]:
import os
os.chdir('/kaggle/working')
!rm -rf sahayak2
!git clone -q https://github.com/shreyaaaah/fewshot-object-detection-for-visually-impaired-person.git sahayak2
%cd /kaggle/working/sahayak2
!pip -q install timm scikit-learn tqdm pyyaml pytest
os.makedirs('/kaggle/temp', exist_ok=True)
RAW = '/kaggle/temp/orbit_raw'              # big downloads (not saved)
ROOT = '/kaggle/temp/orbit_instances'       # converted frames (not saved)
# checkpoints/ and reports/ live in /kaggle/working and ARE saved as notebook output

## 2. Quick self-test (about 2 min). Everything should say passed.

In [ ]:
!python -m pytest -q -p no:cacheprovider tests/test_memory.py tests/test_train_torch.py 2>&1 | tail -3

## 3. See the ORBIT files and their sizes (nothing is downloaded yet)

In [ ]:
import requests, shutil
files = requests.get('https://api.figshare.com/v2/articles/14294597', timeout=60).json()['files']
for f in files:
    print(f"{f['size']/1e9:8.2f} GB  {f['name']}")
want = [f for f in files if '224' in f['name'] and 'unfiltered' not in f['name'].lower()]
total = sum(f['size'] for f in want)
free = shutil.disk_usage('/kaggle/temp').free
print(f"\nWill download {len(want)} files, {total/1e9:.1f} GB total; free disk {free/1e9:.0f} GB")
assert want, 'no *_224 files found - paste the list above to Claude'
assert total + max(f['size'] for f in want) < free - 5e9, 'not enough disk - tell Claude the numbers above'

## 4. Download + unzip the 224 px benchmark frames (one file at a time; can take a while)

In [ ]:
import subprocess
os.makedirs(RAW, exist_ok=True)
for f in want:
    z = '/kaggle/temp/' + f['name']
    print('downloading', f['name'], round(f['size']/1e9, 2), 'GB')
    subprocess.run(['wget', '-q', '-c', '-O', z, f['download_url']], check=True)
    subprocess.run(['unzip', '-q', '-o', z, '-d', RAW], check=True)
    os.remove(z)
print('done')
!du -sh {RAW}

## 5. Convert to `<user>__<object>` folders. First a DRY RUN: check that users and objects look sensible
(expect many users, names like P123__bag, and both 'clean' and 'clutter' listed).

In [ ]:
!python tools/prepare_orbit.py --src {RAW} --out {ROOT} --dry-run

In [ ]:
# real conversion: every 10th frame (ORBIT is 30 fps) keeps variety but stays small
!python tools/prepare_orbit.py --src {RAW} --out {ROOT} --every 10 --max-side 224
!ls {ROOT} | wc -l

## 6. Baselines: no deep learning (A0), frozen ImageNet ResNet-50 (A), frozen DINOv2 (B)

In [ ]:
EV = f'--root {ROOT} --split test --episodes 200 --k-shots 1 3 5 10 --cross-video'
!python -m src.eval.episodic {EV} --frozen-backbone hsv_hist --tag A0_hsv_hist
!python -m src.eval.episodic {EV} --frozen-backbone resnet50.a1_in1k --tag A_resnet50_frozen
!python -m src.eval.episodic {EV} --tag B_dinov2_frozen

## 7. Transfer learning, MAIN model: last 2 blocks + head, open-set prototypical loss (about 10-20 min)

In [ ]:
EPS = 1500
!python -m src.train.train --root {ROOT} --mode partial --loss open --episodes {EPS} --tag C_partial_open --amp
!python -m src.eval.episodic {EV} --ckpt checkpoints/embedder_C_partial_open.pt --tag C_partial_open

## 8. Optional comparisons (run what time allows): LoRA, and the plain prototypical loss

In [ ]:
!python -m src.train.train --root {ROOT} --mode lora --loss open --episodes {EPS} --tag D_lora_open --amp
!python -m src.eval.episodic {EV} --ckpt checkpoints/embedder_D_lora_open.pt --tag D_lora_open

In [ ]:
!python -m src.train.train --root {ROOT} --mode partial --loss proto --episodes {EPS} --tag C_partial_proto --amp
!python -m src.eval.episodic {EV} --ckpt checkpoints/embedder_C_partial_proto.pt --tag C_partial_proto

## 9. Plots and table for the report

In [ ]:
import json, glob, pathlib
import matplotlib.pyplot as plt
R = {}
for f in sorted(glob.glob('reports/episodic_*_test.json')):
    R[pathlib.Path(f).stem.replace('episodic_', '').replace('_test', '')] = json.load(open(f))['results']
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
print(f"{'model':<22}{'K':>3}{'closed-set acc %':>20}{'open-set AUROC':>18}")
for name, res in R.items():
    ks = sorted(int(k) for k in res)
    for a, key, scale, ttl in ((ax[0], 'closed_acc', 100, 'Closed-set accuracy (%) vs shots K'),
                               (ax[1], 'auroc_ours_prob', 1, 'Open-set AUROC (unknown-object rejection) vs K')):
        m = [res[str(k)][key][0] * scale for k in ks]
        c = [res[str(k)][key][1] * scale for k in ks]
        a.errorbar(ks, m, yerr=c, marker='o', capsize=3, label=name)
        a.set_title(ttl); a.set_xlabel('K (examples per object)'); a.grid(alpha=.3)
    for k in ks:
        print(f"{name:<22}{k:>3}{res[str(k)]['closed_acc'][0]*100:>14.1f} +- {res[str(k)]['closed_acc'][1]*100:<4.1f}{res[str(k)]['auroc_ours_prob'][0]:>12.3f}")
ax[0].legend(fontsize=8)
plt.tight_layout(); plt.savefig('reports/orbit_results.png', dpi=150); plt.show()

## 10. Download

In [ ]:
!zip -qr /kaggle/working/orbit_results.zip reports
!cp checkpoints/embedder_C_partial_open.pt /kaggle/working/ 2>/dev/null
# Download from the right panel: Data -> Output -> orbit_results.zip and embedder_C_partial_open.pt